# Neuromorphic SNN vs ANN on CIFAR-10

Ethan Vu and Marcus Vu - CMPE 257

This notebook runs the shared, tested experiment package. The original topology, LIF dynamics and TEBN remain. A stratified validation split chooses checkpoints and the test set is evaluated afterwards. The original exploratory notebook is retained at `SNN_vs_ANN_TEBN_Proposal.ipynb`. Its test-selected best accuracy uses a different evaluation protocol.


## Setup

Clone the full repository before opening this notebook locally. On Kaggle, enable Internet and run `!git clone https://github.com/LittleSA1GON/Neuromorphic-SNN-Experiment.git` once if the repository is not attached. Install `requirements.txt` in a compatible PyTorch/torchvision environment. One visible GPU is used. Choosing two GPUs in Kaggle does not enable multi-GPU training.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "Neuromorphic-SNN-Experiment"]
REPO = next((p for p in candidates if (p / "snn_experiment").is_dir()), None)
if REPO is None:
    raise RuntimeError("Clone or attach the full repository first. See setup.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from snn_experiment.run import Config, run


## Configuration

Start with synthetic tensors to check the installation. Set `SMOKE = False` for full CIFAR-10 training. Each run needs a fresh output folder. Defaults hold out 500 original training examples per class, giving 45,000 training, 5,000 validation and 10,000 test examples.


In [ ]:
from datetime import datetime
SMOKE = True
config = Config(
    model="all", smoke=SMOKE, epochs=1 if SMOKE else 30,
    batch_size=4 if SMOKE else 128, download=not SMOKE,
    data_dir=str(REPO / "data"),
    output=str(REPO / "outputs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")),
    workers=0, seed=42, timesteps=4,
)
results = run(config)


## Results and evidence

Synthetic accuracy checks execution only. `run.json`, `split.json`, `history.csv`, `test_metrics.json`, `operations.json` and `comparison.csv` identify the dataset, protocol and source hashes. Missing GPU power is null. Arithmetic estimates exclude memory, routing and neuron-state overhead. See `docs/evidence_audit.json` for the source-slide audit.


In [ ]:
import pandas as pd
pd.DataFrame(results)


## Optional normalization ablation

Run `Config(model="snn_plain", ...)` in a fresh output folder with matching seed, data and time steps. This baseline shares batch/time statistics without learnable p[t], matching the original notebook. It does not implement BNTT. Train a new model for each T because a TEBN checkpoint has a fixed temporal-scale dimension.
